# Robustheitscheck: Finale Hauptdefinition vs. breite Wildcard-Variante

**Zweck**: Prüfung, ob die **zeitlichen Trends der relativen Verteilungen** stabil bleiben,
wenn statt der finalen Hauptdefinition eine breitere Wildcard-Variante verwendet wird.

**Zeitraum**: 21.04.2022 bis 31.12.2025 (konsistente Scraper-Datenbasis)

**Hauptdefinition (wie in Notebook 03, final)**:
- Klimawandel: wandel, wandels
- Klimakrise: krise, krisen
- Klimaschutz: schutz, schutzes

**Vergleichsvariante (breit)**:
- Klimawandel: wandel*
- Klimakrise: krise*
- Klimaschutz: schutz*

## Setup: Imports & Datenladen

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from datetime import datetime

# Pfade
BASE_PATH = os.path.dirname(os.getcwd())
DATA_OUTPUT = os.path.join(BASE_PATH, 'data_output')
PLOT_DIR = os.path.join(DATA_OUTPUT, 'plots')

# Plots-Ordner erstellen falls nicht vorhanden
os.makedirs(PLOT_DIR, exist_ok=True)

# Daten laden (processed-Versionen)
df_context = pd.read_csv(os.path.join(DATA_OUTPUT, 'dwh_context_processed_2026-01-27.csv'))
df_meta = pd.read_csv(os.path.join(DATA_OUTPUT, 'dwh_meta_processed_2026-01-27.csv'))

print(f"Context-Daten: {len(df_context)} Zeilen")
print(f"Meta-Daten: {len(df_meta)} Zeilen")
print(f"Spalten Context: {df_context.columns.tolist()}")

In [ ]:
# Daten zusammenführen
df = df_context.merge(
    df_meta[['newspaper_id', 'data_published', 'newspaper_name']],
    on='newspaper_id',
    how='left'
)

# Datum konvertieren
df['date'] = pd.to_datetime(df['data_published'])

# Filter: Ab 21.04.2022 (Scraper-Cutoff)
SCRAPER_CUTOFF = '2022-04-21'
df = df[df['date'] >= SCRAPER_CUTOFF].copy()

# Bis end 2025
df = df[df['date'] <= '2025-12-31'].copy()

print(f"Datensatz nach Filter: {len(df)} Nennungen")
print(f"Zeitraum: {df['date'].min()} bis {df['date'].max()}")
print(f"Zeitungen: {df['newspaper_name'].nunique()}")
print(f"\nSpalten in df: {df.columns.tolist()}")

## Hauptdefinition (finale Auswahl wie in 03)

In [ ]:
# Hauptdefinition: exakt die finale Auswahl aus Notebook 03
FINAL_MAPPING = {
    'Klimawandel': ['wandel', 'wandels'],
    'Klimakrise': ['krise', 'krisen'],
    'Klimaschutz': ['schutz', 'schutzes']
}

def map_suffix_main(suffix):
    """Mappt nur die final festgelegten Suffixe auf die drei Begriffe."""
    if pd.isna(suffix):
        return None
    suffix = str(suffix).lower().strip()
    for begriff, suffixe in FINAL_MAPPING.items():
        if suffix in suffixe:
            return begriff
    return None

df['suffix_clean'] = df['suffix'].str.lower().str.strip()
df_v1 = df.copy()
df_v1['Begriff'] = df_v1['suffix_clean'].apply(map_suffix_main)
df_v1 = df_v1[df_v1['Begriff'].notna()].copy()

print("\n=== Hauptdefinition (final, Notebook 03) ===")
print(f"Treffer: {len(df_v1)} Nennungen")
print("\nVerteilung:")
print(df_v1['Begriff'].value_counts())
print("\nFinale Suffixliste:")
for begriff, suffixe in FINAL_MAPPING.items():
    print(f"- {begriff}: {', '.join(suffixe)}")

## Vergleichsvariante: Breite Wildcard-Definition

In [ ]:
# Vergleichsvariante: breite Wildcard-Patterns auf Suffixe
# wandel*, krise*, schutz*
def map_wildcard_suffix(suffix):
    """Mappt breite Wildcard-Suffixe auf die drei Begriffe."""
    if pd.isna(suffix):
        return None

    suffix = str(suffix).lower().strip()
    if suffix.startswith('wandel'):
        return 'Klimawandel'
    if suffix.startswith('krise'):
        return 'Klimakrise'
    if suffix.startswith('schutz'):
        return 'Klimaschutz'
    return None

df_v2 = df.copy()
df_v2['Begriff'] = df_v2['suffix_clean'].apply(map_wildcard_suffix)
df_v2 = df_v2[df_v2['Begriff'].notna()].copy()

print("\n=== Vergleichsvariante (breit: wandel*, krise*, schutz*) ===")
print(f"Treffer: {len(df_v2)} Nennungen")
print("\nVerteilung:")
print(df_v2['Begriff'].value_counts())
print("\nHäufigste Suffix-Varianten (Top 15):")
print(df_v2['suffix_clean'].value_counts().head(15))

In [ ]:
# Zusatz: Welche Varianten kommen in der Wildcard-Variante zusätzlich vor?
konzepte = ['wandel', 'krise', 'schutz']

variant_counts = {}
for k in konzepte:
    counts = df['suffix_clean'].loc[df['suffix_clean'].str.startswith(k, na=False)].value_counts()
    variant_counts[k] = counts

    print(f"\n=== Varianten für '{k}*' (n={counts.sum()}) ===")
    if counts.empty:
        print("Keine Treffer")
    else:
        print(counts.to_string())

varianten_tabelle = pd.concat(
    [
        vc.rename_axis('variante').reset_index(name='haeufigkeit').assign(konzept=k)
        for k, vc in variant_counts.items()
    ],
    ignore_index=True
)[['konzept', 'variante', 'haeufigkeit']]

print("\n=== Gesamttabelle (sortiert) ===")
print(varianten_tabelle.sort_values(['konzept', 'haeufigkeit'], ascending=[True, False]).to_string(index=False))

## Vergleich: Relative Gesamtverteilungen

In [ ]:
# Vergleich der relativen Gesamtverteilungen (nicht absolute Niveaus)
print("\n=== Relative Gesamtverteilung: Hauptdefinition vs. Wildcard ===")
print()

begriffe = ['Klimawandel', 'Klimakrise', 'Klimaschutz']
v1_counts = df_v1['Begriff'].value_counts().reindex(begriffe, fill_value=0)
v2_counts = df_v2['Begriff'].value_counts().reindex(begriffe, fill_value=0)

v1_shares_total = (v1_counts / v1_counts.sum() * 100).round(2)
v2_shares_total = (v2_counts / v2_counts.sum() * 100).round(2)

comparison = pd.DataFrame({
    'Hauptdefinition Anteil (%)': v1_shares_total,
    'Wildcard Anteil (%)': v2_shares_total
})
comparison['Differenz (Pp.)'] = (comparison['Wildcard Anteil (%)'] - comparison['Hauptdefinition Anteil (%)']).round(2)

print(comparison.to_string())
print()
print(f"Absolute Treffer Hauptdefinition: {int(v1_counts.sum())}")
print(f"Absolute Treffer Wildcard: {int(v2_counts.sum())}")
print("Hinweis: Für den Robustheitscheck stehen die relativen Anteile im Fokus.")

## Quartalsvergleich: Relative Anteile und Trendähnlichkeit

In [ ]:
# Quartalweise Aggregation
v1_copy = df_v1.copy()
v1_copy['quarter'] = v1_copy['date'].dt.to_period('Q')
v1_quarterly = v1_copy.groupby(['quarter', 'Begriff']).size().unstack(fill_value=0)

v2_copy = df_v2.copy()
v2_copy['quarter'] = v2_copy['date'].dt.to_period('Q')
v2_quarterly = v2_copy.groupby(['quarter', 'Begriff']).size().unstack(fill_value=0)

# Gleiche Quartale/Begriffe für fairen Vergleich
quarter_index = v1_quarterly.index.union(v2_quarterly.index)
begriff_index = ['Klimawandel', 'Klimakrise', 'Klimaschutz']
v1_quarterly = v1_quarterly.reindex(index=quarter_index, columns=begriff_index, fill_value=0)
v2_quarterly = v2_quarterly.reindex(index=quarter_index, columns=begriff_index, fill_value=0)

# Relative Anteile je Quartal
v1_shares = (v1_quarterly.div(v1_quarterly.sum(axis=1), axis=0) * 100).fillna(0)
v2_shares = (v2_quarterly.div(v2_quarterly.sum(axis=1), axis=0) * 100).fillna(0)

print("\n=== Relative Quartalsanteile: Hauptdefinition ===")
print(v1_shares.round(2).to_string())
print("\n=== Relative Quartalsanteile: Wildcard ===")
print(v2_shares.round(2).to_string())

# Trendähnlichkeit
share_diff = (v2_shares - v1_shares).round(2)
mean_abs_diff = share_diff.abs().mean().round(2)
trend_corr = v1_shares.corrwith(v2_shares).round(3)

print("\n=== Mittlere absolute Abweichung (Pp.) je Begriff ===")
print(mean_abs_diff.to_string())
print("\n=== Korrelation der quartalsweisen Anteils-Trends ===")
print(trend_corr.to_string())

## Grafiken: Relative Quartalsanteile im Vergleich

In [ ]:
# Schwarz-Weiss Plot Setup
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['font.size'] = 10

colors_v1 = {'Klimawandel': '#222222', 'Klimakrise': '#777777', 'Klimaschutz': '#bdbdbd'}
colors_v2 = {'Klimawandel': '#000000', 'Klimakrise': '#5e5e5e', 'Klimaschutz': '#9c9c9c'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for begriff in ['Klimawandel', 'Klimakrise', 'Klimaschutz']:
    ax1.plot(
        range(len(v1_shares)),
        v1_shares[begriff].values,
        marker='o',
        label=begriff,
        color=colors_v1[begriff],
        linewidth=2,
        markersize=4
    )
ax1.set_title('Hauptdefinition: Relative Anteile', fontsize=12, fontweight='bold')
ax1.set_xlabel('Quartal (2022 Q2 bis 2025 Q4)', fontsize=10)
ax1.set_ylabel('Anteil (%)', fontsize=10)
ax1.set_xticks(range(0, len(v1_shares), 2))
ax1.set_xticklabels([str(q) for q in v1_shares.index[::2]], rotation=45, ha='right')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3, linestyle='--')

for begriff in ['Klimawandel', 'Klimakrise', 'Klimaschutz']:
    ax2.plot(
        range(len(v2_shares)),
        v2_shares[begriff].values,
        marker='s',
        label=begriff,
        color=colors_v2[begriff],
        linewidth=2,
        markersize=4
    )
ax2.set_title('Wildcard-Variante: Relative Anteile', fontsize=12, fontweight='bold')
ax2.set_xlabel('Quartal (2022 Q2 bis 2025 Q4)', fontsize=10)
ax2.set_ylabel('Anteil (%)', fontsize=10)
ax2.set_xticks(range(0, len(v2_shares), 2))
ax2.set_xticklabels([str(q) for q in v2_shares.index[::2]], rotation=45, ha='right')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plot_path = os.path.join(PLOT_DIR, '06_robustheitscheck_relative_trends_linien.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Grafik gespeichert: {plot_path}")
plt.show()

## Zusatzprüfung: Zusatzformen und Trenddifferenzen

In [ ]:
print("\n" + "="*80)
print("ZUSATZPRUEFUNG: Wildcard-Zusaetze und Trenddifferenzen")
print("="*80)

# Welche Suffixe kommen nur in der Wildcard-Variante vor?
v1_suffixes_set = set(df_v1['suffix_clean'].dropna().unique())
v2_suffixes_set = set(df_v2['suffix_clean'].dropna().unique())
new_in_v2 = sorted(v2_suffixes_set - v1_suffixes_set)

print(f"\nEindeutige Suffixe Hauptdefinition: {len(v1_suffixes_set)}")
print(f"Eindeutige Suffixe Wildcard: {len(v2_suffixes_set)}")
print(f"Zusatz-Suffixe in Wildcard: {len(new_in_v2)}")

if new_in_v2:
    zusatz_counts = (
        df_v2[df_v2['suffix_clean'].isin(new_in_v2)]['suffix_clean']
        .value_counts()
        .head(20)
    )
    print("\nTop-20 Zusatz-Suffixe (nur Wildcard):")
    print(zusatz_counts.to_string())
else:
    print("Keine Zusatz-Suffixe gefunden.")

print("\nMittlere absolute Quartalsabweichung (Pp.) je Begriff:")
print(mean_abs_diff.to_string())
print("\nKorrelation der Quartalsanteile je Begriff:")
print(trend_corr.to_string())
print("\n" + "="*80)

In [ ]:
# Stacked-Area Plot der relativen Quartalsanteile
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
x_pos = range(len(v1_shares))

ax1.stackplot(
    x_pos,
    v1_shares['Klimawandel'].values,
    v1_shares['Klimakrise'].values,
    v1_shares['Klimaschutz'].values,
    labels=['Klimawandel', 'Klimakrise', 'Klimaschutz'],
    colors=['#333333', '#808080', '#CCCCCC'],
    alpha=0.85
)
ax1.set_title('Hauptdefinition: Relative Anteile', fontsize=12, fontweight='bold')
ax1.set_xlabel('Quartal', fontsize=10)
ax1.set_ylabel('Anteil (%)', fontsize=10)
ax1.set_xticks(range(0, len(v1_shares), 2))
ax1.set_xticklabels([str(q) for q in v1_shares.index[::2]], rotation=45, ha='right')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_ylim([0, 100])
ax1.grid(True, alpha=0.3, axis='y', linestyle='--')

ax2.stackplot(
    x_pos,
    v2_shares['Klimawandel'].values,
    v2_shares['Klimakrise'].values,
    v2_shares['Klimaschutz'].values,
    labels=['Klimawandel', 'Klimakrise', 'Klimaschutz'],
    colors=['#1a1a1a', '#606060', '#a9a9a9'],
    alpha=0.85
)
ax2.set_title('Wildcard-Variante: Relative Anteile', fontsize=12, fontweight='bold')
ax2.set_xlabel('Quartal', fontsize=10)
ax2.set_ylabel('Anteil (%)', fontsize=10)
ax2.set_xticks(range(0, len(v2_shares), 2))
ax2.set_xticklabels([str(q) for q in v2_shares.index[::2]], rotation=45, ha='right')
ax2.legend(loc='upper left', fontsize=9)
ax2.set_ylim([0, 100])
ax2.grid(True, alpha=0.3, axis='y', linestyle='--')

plt.tight_layout()
plot_path = os.path.join(PLOT_DIR, '06_robustheitscheck_relative_trends_flaeche.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Grafik gespeichert: {plot_path}")
plt.show()

## Fazit: Robustheitscheck der Hauptdefinition

In [ ]:
print("\n" + "="*72)
print("FAZIT: Hauptdefinition (final) vs. Wildcard-Variante")
print("="*72)

total_v1 = len(df_v1)
total_v2 = len(df_v2)
difference = total_v2 - total_v1
pct_increase = (difference / total_v1 * 100) if total_v1 > 0 else 0

print("\n1. DEFINITIONEN:")
print("   - Hauptdefinition (final): wandel, wandels, krise, krisen, schutz, schutzes")
print("   - Vergleichsvariante: wandel*, krise*, schutz*")

print("\n2. ABDECKUNG (nur Kontext, nicht Hauptkriterium):")
print(f"   - Hauptdefinition: {total_v1:,} Nennungen")
print(f"   - Wildcard: {total_v2:,} Nennungen")
print(f"   - Differenz: {difference:,} ({pct_increase:.1f}%)")

print("\n3. RELATIVE GESAMTVERTEILUNG (Differenz in Prozentpunkten):")
for begriff in ['Klimawandel', 'Klimakrise', 'Klimaschutz']:
    d = comparison.loc[begriff, 'Differenz (Pp.)']
    print(f"   - {begriff}: {d:+.2f} Pp.")

print("\n4. TRENDAEHNLICHKEIT QUARTALSWEISE (Anteile):")
for begriff in ['Klimawandel', 'Klimakrise', 'Klimaschutz']:
    mad = mean_abs_diff.get(begriff, float('nan'))
    corr = trend_corr.get(begriff, float('nan'))
    print(f"   - {begriff}: mittl. Abweichung {mad:.2f} Pp., Korrelation {corr:.3f}")

if (trend_corr.fillna(0) >= 0.9).all():
    print("\nBewertung: Die relativen Trends bleiben sehr aehnlich (robust).")
elif (trend_corr.fillna(0) >= 0.8).all():
    print("\nBewertung: Die relativen Trends bleiben weitgehend aehnlich (robust).")
else:
    print("\nBewertung: Es gibt erkennbare Trendabweichungen zwischen den Varianten.")

print("\n5. DATENQUALITAET:")
print("   - Zeitfilter konsistent: 21.04.2022 bis 31.12.2025")
print(f"   - Zeitungen im Sample: {df['newspaper_name'].nunique()}")
print("   - Scraper-Umstellung 21.04.2022 beruecksichtigt")

print("\n" + "="*72)